# Comment déplacer un poids ?

Suivre un scalaire depuis sa perte jusqu’au premier pas SGD.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#autograd-et-derivees).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


La perte nous donne maintenant un objectif, mais pas encore une modification des poids. Pour ouvrir ce mécanisme, quittons momentanément les sons. Nous allons régler un seul nombre, avec une cible choisie à la main : 2.

Appelons ce nombre w et choisissons la perte L=(w−2)². À w=0, elle vaut 4 ; à w=2, elle vaut zéro. Ce problème indépendant n’est pas notre couche à deux sorties : il rend seulement un déplacement entièrement calculable.

requires_grad=True demande à PyTorch de suivre les opérations qui dépendent de w. Le point de 0. indique un flottant. Nous pouvons alors calculer la perte et demander sa dérivée, appelée gradient pour notre paramètre.

backward remonte les opérations depuis la perte et écrit la dérivée dans w.grad. Ici, la dérivée est 2(w−2), donc −4 au point zéro. item() extrait ce nombre pour le lire. Le poids, lui, vaut toujours zéro : demander une pente ne réalise pas un déplacement.

La pente négative dit qu’un petit déplacement vers les w plus grands fera baisser cette perte. SGD utilise cette indication avec un taux, noté lr. Avec lr=0,1, sa règle est : nouveau poids = ancien poids − 0,1×gradient.

On retire donc 0,1×(−4) : w augmente de 0,4. Le taux fixe la longueur du déplacement ; le gradient en donne la direction locale. Préparons l’optimiseur qui appliquera exactement ce calcul.

La liste [w] indique quel tenseur ajuster. L’optimiseur garde un accès à ce paramètre et lit son gradient lors de step. Il ne remplace pas w par un nombre Python : notre tenseur suivi reste le paramètre à déplacer.

| Objet | Avant step | Après step |
| --- | --- | --- |
| w | 0 | 0,4 |
| perte déjà calculée | 4 | Toujours 4 |
| perte recalculée au nouveau w | — | 2,56 |

step modifie le paramètre. Il ne réécrit pas rétroactivement les résultats d’une passe précédente.

Revenons à la classification. Sa perte ne suit pas cette petite parabole, et la valeur 2 n’y joue aucun rôle. Le principe reste le même : backward calcule les gradients des vrais poids et biais ; step les déplace. Nous pouvons maintenant répéter cette chaîne sur nos trois extraits.

## Démonstration 1 · Exécuter l’expérience entière

Retrouver les préparations et suivre les valeurs dans l’ordre du programme.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Le flottant w est suivi par autograd ; aucune perte ni dérivée n’est encore calculée.

In [4]:
w = torch.tensor(0., requires_grad=True)


In [5]:
observer('w', w)

w forme () type torch.float32 requires_grad True
  valeurs : 0.0


### Calculer l’état suivant

backward calcule la pente de la perte quadratique ; item lit cette unique dérivée.

In [6]:
perte = (w - 2.) ** 2


In [7]:
observer('perte', perte)

perte forme () type torch.float32 requires_grad True
  valeurs : 4.0


### Calculer les gradients

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [8]:
perte.backward()


In [9]:
observer_entrainement({'w': w, 'perte': perte}, montrer_poids=False)

w.grad forme () type torch.float32 requires_grad False
  valeurs : -4.0


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [10]:
gradient = w.grad.item()


In [11]:
observer('gradient', gradient)

gradient = -4.0


### Choisir les paramètres et leur mise à jour

Nous retenons la valeur précédente pour comparer. step soustrait lr×gradient au paramètre référencé par l’optimiseur.

In [12]:
avant = w.item()
optimiseur = torch.optim.SGD([w], lr=0.1)


In [13]:
observer('avant', avant)

avant = 0.0


### Appliquer une mise à jour

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [14]:
optimiseur.step()


In [15]:
observer_entrainement({'w': w, 'perte': perte, 'gradient': gradient, 'avant': avant, 'optimiseur': optimiseur}, montrer_poids=True)

w forme () type torch.float32 requires_grad True
  valeurs : 0.4000000059604645
w.grad forme () type torch.float32 requires_grad False
  valeurs : -4.0


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [16]:
apres = w.item()


In [17]:
observer('apres', apres)

apres = 0.4000000059604645


### Calculer l’état suivant

L’ancien tenseur perte contient toujours 4. Une nouvelle évaluation de la formule donne le coût du nouveau poids.

In [18]:
perte_recalculee = (w - 2.) ** 2


In [19]:
observer('perte_recalculee', perte_recalculee)

perte_recalculee forme () type torch.float32 requires_grad True
  valeurs : 2.56000018119812


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [20]:
print("gradient :", gradient)
print("poids avant et apres :", avant, apres)
print("ancienne perte :", perte.item())
print("nouvelle perte :", perte_recalculee.item())

gradient : -4.0
poids avant et apres : 0.0 0.4000000059604645
ancienne perte : 4.0
nouvelle perte : 2.56000018119812


In [21]:
assert bool(gradient == -4. and avant == 0.), 'autograd-et-derivees/main: gradient == -4. and avant == 0.'
assert bool(abs(apres - 0.4) < 1e-6 and abs(perte_recalculee.item() - 2.56) < 1e-6), 'autograd-et-derivees/main: abs(apres - 0.4) < 1e-6 and abs(perte_recalculee.item() - 2.56) < 1e-6'
assert bool(perte.item() == 4. and w.grad.item() == -4.), 'autograd-et-derivees/main: perte.item() == 4. and w.grad.item() == -4.'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Prévoyez le gradient de (w−2)² au point w=0.

In [22]:
exo_w = torch.tensor(0.,requires_grad=True)
exo_perte = (exo_w-2.)**2
exo_perte.backward()
exo_gradient = ...  # Remplacer les points par votre réponse.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La dérivée vaut 2×(w−2), évaluée au poids actuel.

</details>

### Exercice 2

Deux backward successifs sur 3u puis 4u, sans remise à zéro : que contient u.grad ?

In [23]:
exo_u = torch.tensor(1.,requires_grad=True)
(3*exo_u).backward()
(4*exo_u).backward()
exo_gradient = ...  # Remplacer les points par votre réponse.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_u.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Les contributions des deux graphes s’accumulent dans le même champ grad.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_w = torch.tensor(0.,requires_grad=True)
exo_perte = (exo_w-2.)**2
exo_perte.backward()
exo_gradient = -4.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_u = torch.tensor(1.,requires_grad=True)
(3*exo_u).backward()
(4*exo_u).backward()
exo_gradient = 7.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_u.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [26]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()